# Human–AI Triage for Hate Speech Moderation — Colab runner

**Before the first run:** upload `cyberhate-triage.zip` to Google Drive in a folder called `cyberhate`, so the path is `MyDrive/cyberhate/cyberhate-triage.zip`.

**How this notebook works**
* Everything (data, splits, predictions, tables) is saved to `MyDrive/cyberhate/`, so nothing is lost if Colab disconnects.
* Every step **resumes**: if a cell is interrupted, reconnect, run **Setup** again, then re-run the interrupted cell. Finished runs are skipped automatically.
* **Part A and Part C use a CPU runtime** (they don't use your free GPU quota). **Part B needs a T4 GPU.** Change with *Runtime → Change runtime type*.
* Free Colab disconnects after ~90 min of inactivity and ~12 h maximum. Keep the tab open; long steps are split into separate cells.
* Each time you change the runtime type, run **Setup** first.

| Part | Runtime | Approx. time |
|---|---|---|
| Data preparation | CPU | 5 min |
| A1 Classical models | CPU | 15–30 min |
| A2 GA search | CPU | 2–4 h |
| A3 BPSO | CPU | 4–8 h (several sessions) |
| B1–B4 Transformers (40 runs) | T4 GPU | ~15–25 min per run |
| B5 Zero-shot LLM | T4 GPU | 2–4 h |
| C Dialect, evaluation, triage, fairness | CPU | 30–60 min |

Times are estimates; the first runs will tell you the real speed.

## Setup — run this every time you connect

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/cyberhate'
os.makedirs(f'{BASE}/raw', exist_ok=True)
os.environ['CH_DATA_DIR'] = f'{BASE}/data'        # splits + processed data (persistent)
os.environ['CH_RESULTS_DIR'] = f'{BASE}/results'  # predictions + tables (persistent)
os.environ['CH_CKPT_DIR'] = '/content/ckpt'       # temporary transformer checkpoints (local disk)

assert os.path.exists(f'{BASE}/cyberhate-triage.zip'), 'Upload cyberhate-triage.zip to MyDrive/cyberhate/ first'
!rm -rf /content/cyberhate-triage && unzip -q -o {BASE}/cyberhate-triage.zip -d /content
%cd /content/cyberhate-triage
!pip install -q vaderSentiment emoji bitsandbytes

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU runtime)')
!nproc

### Optional: smoke test (synthetic data, ~2 min). Confirms the code runs in this Colab environment.

In [ ]:
!python tests/smoke_test.py 2>&1 | tail -3

## Data preparation (CPU, run once)

In [ ]:
# Davidson et al. (full corpus) and HateXplain, from the authors' GitHub repositories
![ -s {BASE}/raw/labeled_data.csv ] || wget -q -O {BASE}/raw/labeled_data.csv https://raw.githubusercontent.com/t-davidson/hate-speech-and-offensive-language/master/data/labeled_data.csv
![ -s {BASE}/raw/hatexplain.json ] || wget -q -O {BASE}/raw/hatexplain.json https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/dataset.json
!ls -la {BASE}/raw

!python run.py prepare --dataset davidson   --davidson-csv {BASE}/raw/labeled_data.csv
!python run.py prepare --dataset hatexplain --hatexplain-json {BASE}/raw/hatexplain.json

Check the printed numbers: Davidson should show **24,783** posts. Copy `n`, `per_class`, `exact_duplicate_texts` and `unanimous_label_pct` into Table I.
If a download failed (file size 0), delete it from `MyDrive/cyberhate/raw/` and re-run the cell.

## Part A — CPU experiments (Runtime type: CPU)

### Quick check first: one split only, ~2 min. Look at the macro-F1 values before spending hours on the rest.

In [ ]:
!python run.py classical --dataset davidson --seeds 42
!python run.py evaluate  --dataset davidson --seeds 42

If LR is roughly in the 0.6–0.75 macro-F1 range, the rebuild is behaving sensibly. Then continue.

### A1. Classical models, all 5 splits, both datasets

In [ ]:
!python run.py classical --dataset davidson
!python run.py classical --dataset hatexplain

### A2. Genetic hyper-parameter search

In [ ]:
!python run.py ga --dataset davidson

In [ ]:
!python run.py ga --dataset hatexplain

### A3. BPSO feature selection (longest CPU step — one cell per run so each fits in a session)

In [ ]:
!python run.py bpso --dataset davidson --base lr_bal 2>&1 | grep -v 'BPSO iter'

In [ ]:
!python run.py bpso --dataset davidson --base mnb 2>&1 | grep -v 'BPSO iter'

In [ ]:
# leakage audit: fitness on the TEST split, only to measure inflation
!python run.py bpso --dataset davidson --base lr_bal --leaky 2>&1 | grep -v 'BPSO iter'

In [ ]:
!python run.py bpso --dataset hatexplain --base lr_bal 2>&1 | grep -v 'BPSO iter'

In [ ]:
!python run.py bpso --dataset hatexplain --base mnb 2>&1 | grep -v 'BPSO iter'

In [ ]:
!python run.py bpso --dataset hatexplain --base lr_bal --leaky 2>&1 | grep -v 'BPSO iter'

## Part B — GPU experiments (Runtime type: **T4 GPU**)
Run **Setup** first after switching. Each cell runs 5 splits × 2 datasets = 10 fine-tuning runs; finished runs are skipped, so if Colab cuts you off, just reconnect and re-run the same cell.

### B1. HateBERT

In [ ]:
!for D in davidson hatexplain; do python run.py transformer --dataset $$D --model hatebert; done

### B2. BERTweet

In [ ]:
!for D in davidson hatexplain; do python run.py transformer --dataset $$D --model bertweet; done

### B3. RoBERTa

In [ ]:
!for D in davidson hatexplain; do python run.py transformer --dataset $$D --model roberta; done

### B4. HateBERT with class-weighted loss

In [ ]:
!for D in davidson hatexplain; do python run.py transformer --dataset $$D --model hatebert --weighted; done

### B5. Zero-shot LLM (Qwen2.5-3B-Instruct, 16-bit)
Each post is scored once and cached in chunks of 1,000, so a disconnect loses at most one chunk (about 10 minutes per chunk on a T4).

In [ ]:
!python run.py llm --dataset davidson --llm Qwen/Qwen2.5-3B-Instruct

In [ ]:
!python run.py llm --dataset hatexplain --llm Qwen/Qwen2.5-3B-Instruct

## Part C — Analysis (Runtime type: CPU)

### C1. Dialect estimation for Davidson (TwitterAAE model of Blodgett et al., 2016)

In [ ]:
![ -d /content/twitteraae ] || git clone -q https://github.com/slanglab/twitteraae /content/twitteraae
!ls /content/twitteraae /content/twitteraae/model | head -20
# the repository's code is Python 2: link its model folder and patch it for Python 3
!ln -sfn /content/twitteraae/model /content/model
!sed -i 's/\.decode("utf-8")//g; s/\.decode(.utf8.)//g; s/\bxrange(/range(/g; s/\.iteritems()/.items()/g; s/\bunicode\b/str/g; s/\bbasestring\b/str/g' /content/twitteraae/code/predict.py
!python run.py dialect --dataset davidson --twitteraae /content/twitteraae

### C2. Evaluation, significance tests, triage simulation, fairness

In [ ]:
for D in ['davidson', 'hatexplain']:
    !python run.py evaluate --dataset {D}
    !python run.py triage   --dataset {D}
    !python run.py fairness --dataset {D}

### C3. Look at the main results

In [ ]:
import pandas as pd
pd.set_option('display.width', 200)
for D in ['davidson', 'hatexplain']:
    s = pd.read_csv(f'{BASE}/results/tables/{D}_summary.csv', index_col=0)
    cols = ['f1_macro_mean', 'f1_macro_std', 'f1_hate_mean', 'recall_hate_mean', 'ece_cal_mean', 'n_seeds']
    print(f'\n===== {D} =====')
    display(s[cols].sort_values('f1_macro_mean', ascending=False).round(4))

### C4. Package everything to send back for writing the paper

In [ ]:
!cd {BASE} && zip -q -r results_for_paper.zip results/tables results/figures data/davidson/description.json data/hatexplain/description.json results/*/*/seed*/meta.json
!ls -la {BASE}/results_for_paper.zip

Download `MyDrive/cyberhate/results_for_paper.zip` and upload it in the chat. The real values then go into the tables and the results text is written from them.